# Korpus 1 — Validierung: Zero-Shot vs. manuelle Kodierung

Vergleicht die Zero-Shot-Klassifikation (Konfiguration aus `Eymann_Methodentest_Korpus1.ipynb`) mit Bernhards finalen Codes aus
`daten/korpus1/Eymann_Korpus1_Validierungsstichprobe_ASSISTENT_VORSCHLAG.csv` (Spalte `vorschlag_modus`, nach Bernhards Korrekturen).

**Wichtig — Limitation (s. `Eymann_Notizen_Methodik.md`, 30.07.2026):** Diese Vergleichsbasis ist teilweise vom Assistenten
vorkodiert und von Bernhard nachträglich korrigiert worden, nicht vollständig unabhängig blind kodiert. Ergebnis entsprechend
vorsichtig interpretieren und im Methodenkapitel als Limitation offenlegen.

**Hinweis:** Dieses Notebook muss lokal laufen (Modell-Download von HuggingFace nötig, im Cowork-Sandbox nicht möglich).

In [1]:
import pandas as pd
from transformers import pipeline

PFAD_VALIDIERUNG = "../daten/korpus1/Eymann_Korpus1_Validierungsstichprobe_ASSISTENT_VORSCHLAG.csv"
manuell = pd.read_csv(PFAD_VALIDIERUNG)
manuell = manuell.rename(columns={"vorschlag_modus": "manueller_modus"})
print(len(manuell), "Segmente geladen")
manuell.head(3)

40 Segmente geladen


,id,anbieter,sprache,text,manueller_modus,begruendung,sicher
0,235,Lawise.ai (Jurilo),en,Take action with confidence Turn answers into ...,zugänglich,"Macht Ergebnisse in klare, umsetzbare Schritte...",nein -- schwaches Signal
1,151,Justement,de,"Von einem solchen Verzicht ist auszugehen, wen...",neutral,"Reines Rechtsprechungs-/Suchergebnis-Zitat, ke...",ja
2,332,balo.ai,de,"Wir bieten Gerichten, Verwaltungen und Organis...",schnell,"Explizit 'schnell, sicher und passgenau'.",ja


## Beers 6 Dimensionen als Hypothesen-Labels (identisch zu `Eymann_Methodentest_Korpus1.ipynb`)

In [2]:
HYPOTHESEN_VORLAGE = {
    "de": "Diese Aussage stellt die Technologie als {} dar.",
    "en": "This statement portrays the technology as {}.",
    "fr": "Cette déclaration présente la technologie comme {}.",
}

BEER_DIMENSIONEN = {
    "schnell": {
        "de": "schnell und zeitsparend",
        "en": "fast and time-saving",
        "fr": "rapide et permettant de gagner du temps",
    },
    "zugänglich": {
        "de": "zugänglich, weil sie komplexe Analysen intuitiv verständlich macht",
        "en": "accessible, making complex analytics intuitive and easy to understand",
        "fr": "accessible, rendant les analyses complexes intuitives et faciles à comprendre",
    },
    "enthüllend": {
        "de": "enthüllend, weil sie objektive Erkenntnisse und verborgene Muster aufdeckt",
        "en": "revealing, uncovering objective insights and hidden patterns",
        "fr": "révélatrice, dévoilant des insights objectifs et des schémas cachés",
    },
    "panoramisch": {
        "de": "panoramisch, mit einem allumfassenden, allsehenden Überblick über die gesamte Datenlandschaft",
        "en": "panoramic, offering an all-seeing view of the entire data landscape",
        "fr": "panoramique, offrant une vue globale et exhaustive de toutes les données",
    },
    "prophetisch": {
        "de": "prophetisch, weil sie zukünftige Entwicklungen und Ergebnisse vorhersagt",
        "en": "prophetic, predicting future developments and outcomes",
        "fr": "prophétique, prédisant les développements et résultats futurs",
    },
    "smart": {
        "de": "smart, weil lernende Algorithmen das Denken selbst übernehmen",
        "en": "smart, with machine-learning algorithms taking on the thinking itself",
        "fr": "intelligente, des algorithmes d'apprentissage prenant en charge la réflexion elle-même",
    },
    "neutral": {
        "de": "neutral, ohne einen besonderen technologischen Vorteil hervorzuheben",
        "en": "neutral, without highlighting any particular technological advantage",
        "fr": "neutre, sans mettre en avant un avantage technologique particulier",
    },
}

def labels_fuer_sprache(sprache):
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return {dim: werte[sprache] for dim, werte in BEER_DIMENSIONEN.items()}

def vorlage_fuer_sprache(sprache):
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return HYPOTHESEN_VORLAGE[sprache]

In [3]:
KONFIDENZ_UNTERGRENZE = 0.3

klassifikator = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ergebnisse = []
for _, row in manuell.iterrows():
    labels_dict = labels_fuer_sprache(row["sprache"])
    label_zu_dimension = {phrase: dim for dim, phrase in labels_dict.items()}
    phrasen = list(labels_dict.values())
    vorlage = vorlage_fuer_sprache(row["sprache"])

    out = klassifikator(row["text"], candidate_labels=phrasen, multi_label=True, hypothesis_template=vorlage)
    bester_index = out["scores"].index(max(out["scores"]))
    bestes_label = out["labels"][bester_index]
    top_dimension = label_zu_dimension[bestes_label]
    top_score = round(out["scores"][bester_index], 3)
    top_dimension_final = "unklar" if top_score < KONFIDENZ_UNTERGRENZE else top_dimension

    ergebnisse.append({
        "id": row["id"],
        "anbieter": row["anbieter"],
        "sprache": row["sprache"],
        "text": row["text"],
        "manueller_modus": row["manueller_modus"],
        "zero_shot_modus": top_dimension_final,
        "zero_shot_score": top_score,
    })

vergleich = pd.DataFrame(ergebnisse)
vergleich.to_csv("../daten/korpus1/Eymann_Korpus1_Validierung_Ergebnis.csv", index=False)
vergleich

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

,id,anbieter,sprache,text,manueller_modus,zero_shot_modus,zero_shot_score
0,235,Lawise.ai (Jurilo),en,Take action with confidence Turn answers into ...,zugänglich,enthüllend,0.363
1,151,Justement,de,"Von einem solchen Verzicht ist auszugehen, wen...",neutral,neutral,0.697
2,332,balo.ai,de,"Wir bieten Gerichten, Verwaltungen und Organis...",schnell,schnell,1.000
3,779,Silex,en,Compliance and Best Practices - Full GDPR comp...,neutral,enthüllend,0.550
4,160,Justement,de,Eventualiter: materielle Haftung wäre ohnehin ...,neutral,zugänglich,0.346
5,726,Lexplorer,de,"Einblicke in die Workbench, Suche und Dokument...",smart,prophetisch,0.799
6,927,Swiss-Noxtua,de,"Alle Funktionen inklusive Einfach testen, ohne...",neutral,schnell,0.902
7,874,Lawcodex,en,Swisslex 5.1 remains available until the end o...,neutral,panoramisch,0.607
8,459,whisperit,en,"By the time I reached the office, my first dra...",schnell,schnell,0.634
9,185,Legartis,en,"Legal expertise, not generic AI Legartis was d...",smart,panoramisch,0.779


## Auswertung: Übereinstimmung

In [4]:
vergleich["uebereinstimmung"] = vergleich["manueller_modus"] == vergleich["zero_shot_modus"]
accuracy = vergleich["uebereinstimmung"].mean()
print(f"Übereinstimmung: {vergleich['uebereinstimmung'].sum()} von {len(vergleich)} ({accuracy:.1%})")
print()
print("Konfusionsmatrix (Zeilen = manuell, Spalten = Zero-Shot):")
print(pd.crosstab(vergleich["manueller_modus"], vergleich["zero_shot_modus"]))
print()
print("Abweichende Fälle:")
abweichungen = vergleich[~vergleich["uebereinstimmung"]][["anbieter", "sprache", "text", "manueller_modus", "zero_shot_modus", "zero_shot_score"]]
for _, r in abweichungen.iterrows():
    print(f"[{r['sprache']}] {r['anbieter']}: manuell={r['manueller_modus']} | Zero-Shot={r['zero_shot_modus']} ({r['zero_shot_score']})")
    print(f"   \"{r['text'][:120]}\"")
    print()

Übereinstimmung: 15 von 40 (37.5%)

Konfusionsmatrix (Zeilen = manuell, Spalten = Zero-Shot):
zero_shot_modus  enthüllend  neutral  panoramisch  prophetisch  schnell  \
manueller_modus                                                           
enthüllend                0        0            0            1        1   
neutral                   2        7            5            1        1   
panoramisch               1        0            1            0        0   
schnell                   0        3            1            0        4   
smart                     0        0            1            1        1   
zugänglich                3        0            0            1        0   

zero_shot_modus  smart  zugänglich  
manueller_modus                     
enthüllend           0           0  
neutral              0           1  
panoramisch          0           1  
schnell              0           0  
smart                3           0  
zugänglich           0           0  

Abweiche

## Für das Methodenkapitel festhalten

- Gesamt-Übereinstimmung (%) und Konfusionsmatrix aus obiger Zelle übernehmen.
- Limitation explizit benennen: Vergleichsbasis ist nicht vollständig unabhängig blind kodiert (s. `Eymann_Notizen_Methodik.md`, 30.07.2026).
- Beobachtete Kategorien-Überlappungen (smart/enthüllend, zugänglich/smart) als eigenständigen, branchenspezifischen Befund einordnen — nicht nur als Kodierunsicherheit.